# 2b — Preprocessing: BTS flight arrivals

**Input:** `data/landing/bts/bts_ontime_YYYY-MM.parquet` (19 files, 1,349,639 rows).
Each file is the monthly Reporting Carrier On-Time Performance extract, already reduced
at download to the fourteen columns in `BTS_COLUMNS` and to flights with an origin *or*
destination in {JFK, LGA, EWR}.

Nineteen files, not eighteen: `FlightDate` is the *departure* date, so an arrival in the
small hours of 1 January 2023 sits in the December 2022 file. `download.py` therefore
fetches BTS from one month before the study window and Step 3 discards everything that
does not land inside it. Step 4 verifies that this actually happened.

**Outputs:**

| File | Contents |
|---|---|
| `data/curated/flights_hourly.parquet` | Table B — one row per `(date, hour, airport)` for JFK and LGA. 26,256 rows, the same grid as Table A. |
| `data/curated/flights_hourly_ewr.parquet` | Newark arrivals on the `(date, hour)` key. 13,128 rows. |
| `data/curated/preprocessing_counts_flights.csv` | Record count after each filter, for the preprocessing table in the report. |
| `data/curated/flight_column_roles.json` | Which columns notebook 2c may use as model features and which it may only use lagged. |

## The contract with the other notebooks

Table A (notebook 2a) is the spine of the join in 2c and is keyed on
`(pickup_date, pickup_hour, airport)`. This notebook writes `(date, hour, airport)`,
because `pickup_date` is meaningless for a flight. Notebook 2c renames Table A's two key
columns to `date` and `hour` on load, then left-joins everything else onto it.

## Two things this notebook is careful about

**The order of the clock transformations is forced.** BTS stores times as `hhmm`
integers, in the *local time of the airport concerned*, against a `FlightDate` that
records the **departure** date. Three corrections are needed — map `2400` to `0`,
advance overnight arrivals by one day, then bucket to the hour — and they must happen in
that order, after the frame has been filtered to arrivals at JFK and LGA. Applied in the
wrong order they produce output that looks entirely reasonable and is wrong for a few
thousand rows a month. Step 2 sets out why.

**Scheduled and realised quantities are kept apart.** A driver deciding at 16:00 whether
to join the JFK queue knows the published schedule. They do not know that the 17:40 from
Chicago will land at 18:25. Every column built from `CRSArrTime` is therefore a legitimate
model feature, and every column built from `ArrTime`, `ArrDelayMinutes`, or `Cancelled` is
an outcome that is only knowable after the fact. The realised columns are still written —
they support a genuinely interesting question about how far delay pulls demand away from
the schedule — but they are declared as such in `flight_column_roles.json` so that 2c can
assert the separation rather than rely on anyone remembering it.

In [1]:
"""Preprocessing of the BTS Reporting Carrier On-Time Performance records.

Reads the raw monthly parquet files, harmonises their schemas, derives a New
York local arrival hour under a strictly ordered set of clock corrections, and
aggregates to the airport-hour grid used by every downstream notebook.
"""

import json
import sys
from datetime import date, timedelta
from functools import reduce
from operator import and_
from pathlib import Path

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

sys.path.append(str(Path("..") / "scripts"))
from spark_utils import (  # noqa: E402
    ARRIVAL_AIRPORTS,
    EWR_AIRPORT,
    count_waterfall,
    create_spark_session,
    hour_spine,
    load_flights,
    print_waterfall,
)

# --- Paths -----------------------------------------------------------------
# Notebook is expected to run from `notebooks/`.
PROJECT_ROOT = Path("..").resolve()
LANDING_DIR = PROJECT_ROOT / "data" / "landing" / "bts"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"
CURATED_DIR.mkdir(parents=True, exist_ok=True)

# --- Study window ----------------------------------------------------------
# Identical to notebook 2a. Inclusive of the start date, exclusive of the end.
WINDOW_START = "2023-01-01"
WINDOW_END = "2024-07-01"
WINDOW_DAYS = 547
EXPECTED_ROWS = WINDOW_DAYS * 24 * len(ARRIVAL_AIRPORTS)  # 26,256, as Table A

# --- Thresholds ------------------------------------------------------------
# BTS defines an on-time arrival as one within 15 minutes of schedule. The same
# threshold is used here so that `share_delayed_15` is comparable with the
# published on-time statistics rather than being a private definition.
DELAY_THRESHOLD_MIN = 15

# Great-circle miles above which a flight is treated as long haul. The
# conventional transcontinental cutoff: flights beyond it are flown by larger
# aircraft, and their passengers carry more luggage and have less appetite for
# the AirTrain, both of which should raise the taxi share of arrivals.
LONGHAUL_MI = 1500

In [2]:
spark = create_spark_session(app_name="MAST30034 — flight preprocessing")
spark.sparkContext.setLogLevel("WARN")
spark.version

your 131072x1 screen size is bogus. expect trouble
26/08/16 19:58:40 WARN Utils: Your hostname, iphone resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/16 19:58:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/16 19:58:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.1'

## Step 1 — Load and harmonise the monthly files

`load_flights` mirrors `load_trips`: it reads every `bts_ontime_*.parquet` file, passes
each through `normalise_bts_schema`, and unions them with `unionByName`.

Normalisation matters more here than on the taxi side. `download.py` writes each month
with pandas, which infers dtypes per file, so a month in which nothing was cancelled
stores `Cancelled` as an integer while every other month stores it as a float. `ArrTime`
and `ArrDelayMinutes` drift the same way, because both are null for cancelled flights and
a column of integers with one null becomes a column of floats. Unioning the raw frames
fails on type mismatch. Column names are also lower-cased so that the flight and taxi
tables follow one convention.

In [3]:
flights_raw = load_flights(spark, LANDING_DIR)
raw_total = flights_raw.count()

print(f"{raw_total:,} rows, {len(flights_raw.columns)} columns after normalisation")
print(flights_raw.columns)

1,349,639 rows, 14 columns after normalisation
['flightdate', 'reporting_airline', 'flight_number_reporting_airline', 'origin', 'dest', 'crsdeptime', 'deptime', 'depdelayminutes', 'crsarrtime', 'arrtime', 'arrdelayminutes', 'cancelled', 'diverted', 'distance']


## Step 2 — Arrivals only, then the clock corrections

### Why the destination filter comes first

The landing files contain every flight that *touches* an NYC airport, in either
direction. Roughly half are departures, for which `CRSArrTime` is the local time at some
other airport entirely: on a row with `Origin = LGA` and `Dest = DEN`, `CRSArrTime` is
Denver time. Deriving an hour before applying the destination filter would compute a New
York hour from a Denver clock reading for half the frame.

Filtering to `Dest ∈ {JFK, LGA}` first buys a second guarantee that the overnight
correction below depends on. Every domestic origin lies at or west of Eastern time, so an
inbound flight's local clock cannot run backwards: the elapsed wall-clock time between
departure and arrival is always non-negative. The one exception is Puerto Rico and the US
Virgin Islands, which sit an hour east of Eastern in winter — but those flights are more
than three hours long, so their clock difference stays comfortably positive.

### The three corrections, in order

| Order | Correction | Why it must come here |
|---|---|---|
| 1 | `2400 → 0` | BTS writes midnight as `2400`. Left as-is it buckets to hour 24, which does not exist. |
| 2 | `+1 day where CRSArrTime < CRSDepTime` | `FlightDate` is the **departure** date. A flight leaving at 22:30 and landing at 00:45 lands on the following calendar day. |
| 3 | Bucket to the hour | Integer division by 100. |

The order of 1 and 2 is the part that is easy to get wrong. Take a red-eye scheduled to
depart at 22:30 and arrive at `2400`. Do the overnight comparison first and `2400 > 2230`,
so no day is added, and after mapping `2400 → 0` the flight is recorded as landing at
midnight *at the start of* its departure day — twenty-two and a half hours early. Map
`2400 → 0` first and the comparison becomes `0 < 2230`, the day is advanced, and the
flight lands at 00:00 the next morning, which is what `2400` means. Doing the mapping
first makes the overnight rule handle midnight for free, with no special case.

`add_arrival_key` implements all three in one place and is used three times: for the
scheduled key, for the realised key, and for Newark.

In [4]:
arrivals = flights_raw.where(F.col("dest").isin(list(ARRIVAL_AIRPORTS)))
arrivals_total = arrivals.count()

print(f"{arrivals_total:,} arrivals at {', '.join(ARRIVAL_AIRPORTS)} "
      f"({100 * arrivals_total / raw_total:.1f}% of the landing files)")
arrivals.groupBy("dest").count().orderBy("dest").show()

461,205 arrivals at JFK, LGA (34.2% of the landing files)
+----+------+
|dest| count|
+----+------+
| JFK|204135|
| LGA|257070|
+----+------+



In [5]:
def valid_hhmm(column: str):
    """Predicate: `column` holds a well-formed BTS `hhmm` clock reading.

    A reading is well formed if it is non-null, lies in [0, 2400], and has a
    minute component below 60. Values outside this set cannot be bucketed into
    an hour and are removed rather than repaired, since there is no way to know
    which digit is wrong.

    Args:
        column: Name of an integer `hhmm` column.

    Returns:
        A boolean Column.
    """
    return (
        F.col(column).isNotNull()
        & (F.col(column) >= 0)
        & (F.col(column) <= 2400)
        & (F.col(column) % 100 < 60)
    )


def add_arrival_key(
    df: DataFrame,
    date_col: str,
    arr_time_col: str,
    dep_time_col: str,
    prefix: str,
) -> DataFrame:
    """Derive a New York local arrival date and hour from BTS clock fields.

    Applies the three corrections in the order set out above: midnight is
    remapped, the date is advanced for overnight arrivals, and the result is
    bucketed to the hour. The caller must have filtered the frame to arrivals
    at a New York airport first, or the derived hour is in the wrong timezone.

    Args:
        df: Frame of arrivals, already filtered by destination.
        date_col: Column holding the departure date (`flightdate`).
        arr_time_col: Arrival clock column, `hhmm` (`crsarrtime` or `arrtime`).
        dep_time_col: Departure clock column, `hhmm`, used only for the
            overnight comparison.
        prefix: Prefix for the three derived columns.

    Returns:
        The frame with `{prefix}_hhmm`, `{prefix}_date`, and `{prefix}_hour`.
    """
    # 1. Midnight. Applied to both clock fields so that the comparison in
    #    step 2 sees midnight as the start of a day rather than the end of one.
    arr_hhmm = F.when(F.col(arr_time_col) == 2400, F.lit(0)) \
                .otherwise(F.col(arr_time_col))
    dep_hhmm = F.when(F.col(dep_time_col) == 2400, F.lit(0)) \
                .otherwise(F.col(dep_time_col))

    df = df.withColumn(f"{prefix}_hhmm", arr_hhmm) \
           .withColumn(f"_{prefix}_dep_hhmm", dep_hhmm)

    # 2. Overnight arrivals land on the day after FlightDate.
    df = df.withColumn(
        f"{prefix}_date",
        F.when(
            F.col(f"{prefix}_hhmm") < F.col(f"_{prefix}_dep_hhmm"),
            F.date_add(F.col(date_col), 1),
        ).otherwise(F.col(date_col)),
    )

    # 3. Hour bucket.
    df = df.withColumn(
        f"{prefix}_hour", F.floor(F.col(f"{prefix}_hhmm") / 100).cast("int")
    )

    return df.drop(f"_{prefix}_dep_hhmm")

In [6]:
keyed = arrivals.withColumn("flightdate", F.to_date("flightdate"))

# Scheduled key: what the arrivals board said, and the basis of every model
# feature. Realised key: when the aircraft actually reached the gate.
keyed = add_arrival_key(keyed, "flightdate", "crsarrtime", "crsdeptime", "sched")
keyed = add_arrival_key(keyed, "flightdate", "arrtime", "deptime", "actual")

keyed = keyed.withColumn("airport", F.col("dest")).cache()

keyed.select(
    "flightdate", "crsdeptime", "crsarrtime", "sched_date", "sched_hour",
    "arrtime", "actual_date", "actual_hour", "cancelled",
).show(5, truncate=False)

+----------+----------+----------+----------+----------+-------+-----------+-----------+---------+
|flightdate|crsdeptime|crsarrtime|sched_date|sched_hour|arrtime|actual_date|actual_hour|cancelled|
+----------+----------+----------+----------+----------+-------+-----------+-----------+---------+
|2022-12-19|523       |630       |2022-12-19|6         |624    |2022-12-19 |6          |0.0      |
|2022-12-20|523       |630       |2022-12-20|6         |615    |2022-12-20 |6          |0.0      |
|2022-12-21|523       |630       |2022-12-21|6         |616    |2022-12-21 |6          |0.0      |
|2022-12-22|523       |630       |2022-12-22|6         |611    |2022-12-22 |6          |0.0      |
|2022-12-23|523       |630       |2022-12-23|6         |NULL   |2022-12-23 |NULL       |1.0      |
+----------+----------+----------+----------+----------+-------+-----------+-----------+---------+
only showing top 5 rows



### Is the overnight correction right?

Not "does it look right" — the correction can be shown to be exact, and the diagnostics
below verify the one assumption the proof rests on.

Write `d` for the scheduled departure minute-of-day at the origin and `B` for the block
time plus the eastward timezone gain, both in minutes. The New York arrival
minute-of-day is then `a = (d + B) mod 1440`, and the arrival genuinely falls on the day
after `FlightDate` exactly when `d + B ≥ 1440`. The rule applied above shifts when
`a < d`. For any `0 < B < 1440` these are the same statement:

`a < d` ⟺ `(d + B) mod 1440 < d` ⟺ `d + B ≥ 1440`

So the correction is exact, not heuristic, provided no leg blocks a full day of clock
time. The longest domestic arrival into New York — Honolulu, roughly eleven hours of
flying plus a five or six hour timezone gain — implies about sixteen hours, well inside
the bound. Nothing in this feed can violate it.

**What the diagnostics can and cannot show.** Because `implied_block = 1440·shift + a − d`,
it lies in `[0, 1440)` by construction. A wrongly fired shift would wrap back into that
range rather than exceeding it, so this statistic *cannot* detect one — a fact worth
stating, because a bounded quantity that never looks alarming is easily mistaken for
evidence. What it does detect is the case the proof does not cover: a corrupt clock
field. A row implying eighteen hours of clock time is not a long flight, it is a wrong
number, and those rows are counted and shown rather than asserted away, because a
handful of bad rows in an external feed is a data-quality note, not a reason to halt a
marker's re-run.

What *is* asserted is the invariant itself. If `implied_block` ever falls outside
`[0, 1440)`, the arithmetic in this notebook is broken — most likely the `2400` remap
having been applied to one clock field and not the other — and everything downstream is
wrong.

In [7]:
# Longest clock-time block a real domestic arrival into New York can imply:
# about eleven hours from Honolulu plus a six-hour timezone gain. Rounded up to
# eighteen hours so that the genuine Hawaii and Alaska services sit comfortably
# inside it and only corrupt clock fields fall outside.
MAX_PLAUSIBLE_BLOCK_MIN = 18 * 60

# Both clock fields go through the same 2400 remap. Using the raw `crsdeptime`
# here while the arrival side uses the remapped `sched_hhmm` would make the two
# minute counts incomparable and could produce a negative block.
dep_hhmm = F.when(F.col("crsdeptime") == 2400, F.lit(0)).otherwise(F.col("crsdeptime"))

sched_minutes = (
    F.floor(F.col("sched_hhmm") / 100) * 60 + F.col("sched_hhmm") % 100
)
dep_minutes = F.floor(dep_hhmm / 100) * 60 + dep_hhmm % 100
implied_block = (
    F.datediff(F.col("sched_date"), F.col("flightdate")) * 1440
    + sched_minutes - dep_minutes
)

well_formed = keyed.where(valid_hhmm("crsarrtime") & valid_hhmm("crsdeptime"))
diagnostics = well_formed.agg(
    F.count("*").alias("rows_examined"),
    F.avg(F.when(F.col("sched_date") != F.col("flightdate"), 1.0).otherwise(0.0))
    .alias("share_shifted_overnight"),
    F.min(implied_block).alias("min_block_min"),
    F.percentile_approx(implied_block, 0.5).alias("median_block_min"),
    F.max(implied_block).alias("max_block_min"),
    F.sum(
        F.when(implied_block > MAX_PLAUSIBLE_BLOCK_MIN, 1).otherwise(0)
    ).alias("implausible_block_rows"),
    F.sum(F.when(F.col("crsarrtime") == 2400, 1).otherwise(0))
    .alias("crsarrtime_2400_rows"),
).collect()[0]

for field, value in diagnostics.asDict().items():
    print(f"{field:<28} {value:,.4f}" if isinstance(value, float)
          else f"{field:<28} {value:,}")

# The invariant. A violation means the arithmetic above is wrong, not that the
# data is unusual, so this one does stop the run.
assert 0 <= diagnostics["min_block_min"], "Negative implied block time"
assert diagnostics["max_block_min"] < 1440, "Implied block exceeds a full day"

# The data-quality figure, reported rather than enforced.
implausible = diagnostics["implausible_block_rows"]
share = implausible / diagnostics["rows_examined"]
print(f"\nImplied block beyond {MAX_PLAUSIBLE_BLOCK_MIN // 60} h: "
      f"{implausible:,} rows ({share:.4%}) — corrupt clock fields, not long flights.")

if implausible:
    print("\nThe offending rows, for the data-quality note in the report:")
    (
        well_formed
        .where(implied_block > MAX_PLAUSIBLE_BLOCK_MIN)
        .select(
            "flightdate", "reporting_airline", "origin", "dest",
            "crsdeptime", "crsarrtime", "sched_date", "sched_hour",
            implied_block.alias("implied_block_min"),
        )
        .orderBy(F.desc("implied_block_min"))
        .show(10, truncate=False)
    )

rows_examined                461,205
share_shifted_overnight      0.0479
min_block_min                24
median_block_min             161
max_block_min                1,323
implausible_block_rows       2
crsarrtime_2400_rows         0

Implied block beyond 18 h: 2 rows (0.0004%) — corrupt clock fields, not long flights.

The offending rows, for the data-quality note in the report:
+----------+-----------------+------+----+----------+----------+----------+----------+-----------------+
|flightdate|reporting_airline|origin|dest|crsdeptime|crsarrtime|sched_date|sched_hour|implied_block_min|
+----------+-----------------+------+----+----------+----------+----------+----------+-----------------+
|2024-01-20|YX               |MSN   |LGA |1123      |926       |2024-01-21|9         |1323             |
|2023-02-18|YX               |IND   |LGA |1030      |810       |2023-02-19|8         |1300             |
+----------+-----------------+------+----+----------+----------+----------+----------+-----

## Step 3 — Filters and record counts

The destination filter has already been applied, physically, because everything after it
depends on it. It is reported as the first stage of the waterfall so the table in the
report reads as one sequence, and the two counts are cross-checked below.

| # | Filter | Rule being enforced |
|---|---|---|
| 1 | Arrival at JFK or LGA | Only arrivals generate airport pickups, and only these two airports lie in a zone where a yellow taxi may pick up. Applying it first also guarantees that the remaining clock fields are New York local. |
| 2 | Well-formed scheduled clock fields | `CRSArrTime` and `CRSDepTime` must be non-null `hhmm` values in `[0, 2400]` with a minute component below 60. A malformed reading cannot be bucketed into an hour. |
| 3 | Plausible implied block time | A row whose scheduled arrival precedes its scheduled departure by hours on a short domestic leg has at least one corrupt clock field, and the overnight correction pushes it to the wrong day. Step 2 identifies these; this removes them rather than leaving a known-bad row in the table. |
| 4 | Scheduled arrival inside the study window | Applied **after** the overnight shift, so a flight departing 30 June 2024 and landing on 1 July is excluded on its arrival date rather than its departure date, and one departing 31 December 2022 and landing on 1 January 2023 is included. |

Cancelled and diverted flights are **retained** here. A cancellation is visible on the
arrivals board, so it is part of what a driver can see when deciding where to queue, and
removing it would misstate the scheduled series. Both are excluded from the realised
series in Step 5, where they would instead misstate what actually landed.

In [8]:
FILTERS = [
    (
        "Well-formed scheduled clock fields",
        valid_hhmm("crsarrtime") & valid_hhmm("crsdeptime"),
    ),
    (
        # `implied_block` is defined in Step 2, where these rows are listed.
        # Finding them and then keeping them would be worse than not looking.
        "Plausible implied block time",
        implied_block <= MAX_PLAUSIBLE_BLOCK_MIN,
    ),
    (
        "Scheduled arrival within study window",
        (F.col("sched_date") >= F.lit(WINDOW_START).cast("date"))
        & (F.col("sched_date") < F.lit(WINDOW_END).cast("date")),
    ),
]

stages = count_waterfall(keyed, FILTERS)

# `count_waterfall` labels its stage 0 as the raw ingest, which here is the
# already-filtered arrivals frame. Relabel it and prepend the true raw count so
# the table matches the report.
assert stages[0][1] == arrivals_total, "Arrival count disagrees with Step 2"
waterfall = [
    ("Raw BTS ingest", raw_total, None),
    ("Arrival at JFK or LGA", arrivals_total, raw_total - arrivals_total),
] + stages[1:]

print_waterfall(waterfall)

Raw BTS ingest                                1,349,639  removed          —  (100.00% of raw)
Arrival at JFK or LGA                           461,205  removed    888,434  ( 34.17% of raw)
Well-formed scheduled clock fields              461,205  removed          0  ( 34.17% of raw)
Plausible implied block time                    461,203  removed          2  ( 34.17% of raw)
Scheduled arrival within study window           436,754  removed     24,449  ( 32.36% of raw)


In [9]:
# Persist for the report table. Retention is reported against the raw ingest.
rows = [
    {
        "step": label,
        "rows": remaining,
        "removed": removed,
        "pct_of_raw": round(100 * remaining / raw_total, 3),
    }
    for label, remaining, removed in waterfall
]

counts_path = CURATED_DIR / "preprocessing_counts_flights.csv"
spark.createDataFrame(rows).coalesce(1).toPandas().to_csv(counts_path, index=False)
print(f"Written to {counts_path}")

Written to /home/tavish/projects/project-1-individual-SavvyHack/data/curated/preprocessing_counts_flights.csv


### Both ends of the window

`FlightDate` is the departure date, so the study window has a soft edge at each end and
they need opposite treatment.

**The opening edge is a gap to be filled.** A red-eye that left the west coast on 31
December 2022 lands on 1 January 2023 and belongs in the window, but it is recorded in
the December 2022 file. `download.py` fetches that month for exactly this reason, and
filter 4 then keeps only the handful of its rows that land inside the window. The check
below counts them, and fails outright if the preceding month is missing from the landing
directory — otherwise the recovery would silently do nothing for anyone who downloaded
only the eighteen months of the study and the first morning of the data would be short by
roughly ten percent without any sign of it.

**The closing edge is a genuine exclusion.** Flights in the June 2024 file that were
shifted past `WINDOW_END` land on 1 July, outside the study, and are correctly dropped.
They are counted only so the report can state the size of the effect at both ends
symmetrically.

In [10]:
# The month before the window must be on disk, or the opening-edge recovery
# below is a no-op that leaves the first morning quietly under-counted.
window_start_date = date.fromisoformat(WINDOW_START)
preceding_month = (window_start_date - timedelta(days=1)).strftime("%Y-%m")
preceding_file = LANDING_DIR / f"bts_ontime_{preceding_month}.parquet"
assert preceding_file.exists(), (
    f"{preceding_file.name} is missing. Overnight arrivals landing on "
    f"{WINDOW_START} are recorded in that month and would be lost. Run: "
    f"python scripts/download.py --start {preceding_month} "
    f"--end {preceding_month} --skip-taxi --skip-weather"
)

recovered = keyed.where(
    (F.col("flightdate") < F.lit(WINDOW_START).cast("date"))
    & (F.col("sched_date") >= F.lit(WINDOW_START).cast("date"))
).count()

rolled_past_close = keyed.where(
    valid_hhmm("crsarrtime")
    & valid_hhmm("crsdeptime")
    & (F.col("sched_date") >= F.lit(WINDOW_END).cast("date"))
).count()

first_day_arrivals = keyed.where(
    F.col("sched_date") == F.lit(WINDOW_START).cast("date")
).count()

print(f"Recovered from {preceding_month}, landing on {WINDOW_START}: {recovered:,}")
print(f"Arrivals on the first day of the window:            {first_day_arrivals:,}")
print(f"  of which recovered:                               "
      f"{100 * recovered / max(first_day_arrivals, 1):.1f}%")
print(f"Dropped at the closing edge, landing on {WINDOW_END}:  {rolled_past_close:,}")

assert recovered > 0, "No arrivals recovered from the preceding month"

Recovered from 2022-12, landing on 2023-01-01: 24
Arrivals on the first day of the window:            516
  of which recovered:                               4.7%
Dropped at the closing edge, landing on 2024-07-01:  49


In [11]:
flights = keyed.where(reduce(and_, (predicate for _, predicate in FILTERS))).cache()
flights_total = flights.count()

assert flights_total == waterfall[-1][1], (
    f"Filtered {flights_total:,} rows but waterfall predicted {waterfall[-1][1]:,}"
)
print(f"{flights_total:,} in-window scheduled arrivals")

flights.agg(
    F.sum(F.when(F.col("cancelled") == 1, 1).otherwise(0)).alias("cancelled"),
    F.sum(F.when(F.col("diverted") == 1, 1).otherwise(0)).alias("diverted"),
    F.sum(F.when(F.col("arrtime").isNull(), 1).otherwise(0)).alias("null_arrtime"),
    F.sum(F.when(F.col("arrdelayminutes").isNull(), 1).otherwise(0))
    .alias("null_arrdelay"),
).show()

436,754 in-window scheduled arrivals
+---------+--------+------------+-------------+
|cancelled|diverted|null_arrtime|null_arrdelay|
+---------+--------+------------+-------------+
|    10389|    1634|       10631|        12023|
+---------+--------+------------+-------------+



## Step 4 — The scheduled series

One row per `(sched_date, sched_hour, airport)`, aggregated over the flights the arrivals
board showed for that hour. Every column here is knowable in advance and is therefore
admissible as a model feature.

- **`sched_arrivals`** — the count. The primary flight-side predictor.
- **`sched_distance_mi`** — summed great-circle distance. A crude proxy for aircraft size,
  and therefore for passengers: a widebody from the west coast delivers several times the
  taxi demand of a regional jet from Buffalo, and BTS gives no seat count.
- **`share_longhaul`** — the share of the hour's arrivals flying at least
  `LONGHAUL_MI`. Distinguishes an hour of many small flights from an hour of few large
  ones, which the sum alone cannot.

Three further columns are realised outcomes, aggregated on the same key because they
describe the cohort *scheduled* for that hour — how late that cohort ran, and how much of
it never came:

- **`share_cancelled`**, over all scheduled arrivals in the hour.
- **`mean_arr_delay_min`** and **`share_delayed_15`**, over the arrivals whose delay was
  actually observed.

That last denominator is carried explicitly as `n_delay_observed` rather than being
inferred. Cancelled *and* diverted flights both have a null `ArrDelayMinutes`, so
subtracting cancellations from the scheduled count gives the wrong denominator for any
hour containing a diversion, and an hour whose only arrival was diverted would otherwise
report a null mean delay alongside a zero cancellation rate — a combination that looks
like a bug in the aggregation when it is in fact the data.

In [12]:
delay_observed = F.col("arrdelayminutes").isNotNull()

sched_hourly = flights.groupBy(
    F.col("sched_date").alias("date"),
    F.col("sched_hour").alias("hour"),
    "airport",
).agg(
    F.count("*").alias("sched_arrivals"),
    F.sum("distance").alias("sched_distance_mi"),
    F.avg(F.when(F.col("distance") >= LONGHAUL_MI, 1.0).otherwise(0.0))
    .alias("share_longhaul"),
    F.avg(F.when(F.col("cancelled") == 1, 1.0).otherwise(0.0))
    .alias("share_cancelled"),
    # Denominator for the two delay columns, carried explicitly.
    F.count("arrdelayminutes").alias("n_delay_observed"),
    F.avg("arrdelayminutes").alias("mean_arr_delay_min"),
    F.avg(
        F.when(
            delay_observed,
            F.when(F.col("arrdelayminutes") >= DELAY_THRESHOLD_MIN, 1.0)
            .otherwise(0.0),
        )
    ).alias("share_delayed_15"),
)

print(f"{sched_hourly.count():,} non-empty scheduled airport-hours")

20,914 non-empty scheduled airport-hours


## Step 5 — The realised series

The same flights, keyed instead on when they actually landed. Cancelled flights never
landed and are excluded; so are diversions, which by definition arrived somewhere else,
even where BTS records a later arrival time for the eventual continuation.

Holding `sched_arrivals` and `actual_arrivals` side by side on one row is what makes the
delay question answerable. Their difference, `arrival_slippage`, is the number of aircraft
an hour gained or lost against the board, and it is the right diagnostic for asking how
much a schedule-based prediction is disrupted by delay — a question worth a paragraph in
the report whichever way it comes out.

In [13]:
realised = flights.where(
    (F.coalesce(F.col("cancelled"), F.lit(0.0)) == 0)
    & (F.coalesce(F.col("diverted"), F.lit(0.0)) == 0)
    & valid_hhmm("arrtime")
    & valid_hhmm("deptime")
    & (F.col("actual_date") >= F.lit(WINDOW_START).cast("date"))
    & (F.col("actual_date") < F.lit(WINDOW_END).cast("date"))
)
realised_total = realised.count()

realised_hourly = realised.groupBy(
    F.col("actual_date").alias("date"),
    F.col("actual_hour").alias("hour"),
    "airport",
).agg(F.count("*").alias("actual_arrivals"))

print(f"{realised_total:,} arrivals actually landed in the window "
      f"({100 * realised_total / flights_total:.2f}% of scheduled)")

424,670 arrivals actually landed in the window (97.23% of scheduled)


## Step 6 — The spine, the join, and the neighbouring-hour features

An airport-hour with no scheduled arrivals produces no group and is absent from both
aggregations above. Those hours are real — JFK schedules nothing to land at 03:00 most
nights — and they are exactly the hours a driver most needs the model to get right, so a
complete grid is manufactured and the aggregations are left-joined onto it. This is the
same construction as Table A, from the same helper, which is what guarantees the two
tables share a key.

Counts and sums are filled with zero: no scheduled arrivals really does mean zero
arrivals and zero miles. The share and mean columns are left **null**, because the mean
delay of no flights is undefined rather than zero, and filling it would plant a spurious
cluster at the origin in every plot that touches delay.

The neighbouring-hour counts are computed after the join, over the completed grid, so
that `lag` genuinely means "the previous hour" rather than "the previous hour that
happened to have flights". Both are legitimate features: the 17:00 schedule is published
well before a driver decides whether to queue at 16:00. They are null at the two ends of
the window, which the modelling notebook must handle.

Daylight saving is not re-flagged here. Table A already carries `dst_anomaly` for the six
affected rows, and it survives the join in 2c; duplicating the flag would create two
sources of truth for one fact.

In [14]:
spine = hour_spine(spark, WINDOW_START, WINDOW_END, airports=ARRIVAL_AIRPORTS)
assert spine.count() == EXPECTED_ROWS, "Spine does not cover the study window"

ZERO_FILL = ["sched_arrivals", "sched_distance_mi", "n_delay_observed",
             "actual_arrivals"]

table_b = (
    spine
    .join(sched_hourly, ["date", "hour", "airport"], how="left")
    .join(realised_hourly, ["date", "hour", "airport"], how="left")
    .fillna(0, subset=ZERO_FILL)
    .withColumn(
        "arrival_slippage", F.col("actual_arrivals") - F.col("sched_arrivals")
    )
)

assert table_b.count() == EXPECTED_ROWS, "A join changed the row count"

In [15]:
# Ordered over the completed grid, so `lag` is the previous clock hour.
neighbours = Window.partitionBy("airport").orderBy("date", "hour")

table_b = (
    table_b
    .withColumn("sched_arrivals_prev_hr", F.lag("sched_arrivals").over(neighbours))
    .withColumn("sched_arrivals_next_hr", F.lead("sched_arrivals").over(neighbours))
)

table_b.orderBy("date", "hour", "airport").show(6, truncate=False)

+----------+----+-------+--------------+-----------------+--------------+---------------+----------------+------------------+----------------+---------------+----------------+----------------------+----------------------+
|date      |hour|airport|sched_arrivals|sched_distance_mi|share_longhaul|share_cancelled|n_delay_observed|mean_arr_delay_min|share_delayed_15|actual_arrivals|arrival_slippage|sched_arrivals_prev_hr|sched_arrivals_next_hr|
+----------+----+-------+--------------+-----------------+--------------+---------------+----------------+------------------+----------------+---------------+----------------+----------------------+----------------------+
|2023-01-01|0   |JFK    |0             |0.0              |NULL          |NULL           |0               |NULL              |NULL            |0              |0               |NULL                  |1                     |
|2023-01-01|0   |LGA    |0             |0.0              |NULL          |NULL           |0               |NULL  

## Step 7 — Newark

Newark is not a taxi zone a yellow cab may pick up from, so it never appears in the
target. It matters as context: it draws from the same pool of arriving passengers and the
same weather systems, and a disrupted evening at EWR pushes rebooked travellers towards
the other two airports. It is built on the `(date, hour)` key alone, with an `ewr_` prefix
so 2c can join it without column collisions.

The scheduled columns are the same three as Table B. The realised columns are kept to
cancellations and mean delay: Newark is a covariate, not a subject, and every extra column
here is another chance for the model to overfit citywide noise.

In [16]:
ewr = flights_raw.where(F.col("dest") == EWR_AIRPORT).withColumn(
    "flightdate", F.to_date("flightdate")
)
ewr = add_arrival_key(ewr, "flightdate", "crsarrtime", "crsdeptime", "sched").where(
    valid_hhmm("crsarrtime")
    & valid_hhmm("crsdeptime")
    & (F.col("sched_date") >= F.lit(WINDOW_START).cast("date"))
    & (F.col("sched_date") < F.lit(WINDOW_END).cast("date"))
)

ewr_hourly = ewr.groupBy(
    F.col("sched_date").alias("date"), F.col("sched_hour").alias("hour")
).agg(
    F.count("*").alias("ewr_sched_arrivals"),
    F.sum("distance").alias("ewr_sched_distance_mi"),
    # Defined exactly as in Table B: a cancelled or diverted flight is part of
    # the schedule but contributes no delay reading, so it counts towards
    # `ewr_sched_arrivals` and not towards the mean. Writing this differently
    # here would give the report two silently different definitions of delay.
    F.sum(
        F.when(
            (F.col("cancelled") == 0) & F.col("arrdelayminutes").isNotNull(), 1
        ).otherwise(0)
    ).alias("ewr_n_delay_observed"),
    F.avg(
        F.when(F.col("cancelled") == 0, F.col("arrdelayminutes"))
    ).alias("ewr_mean_arr_delay_min"),
    F.avg(F.when(F.col("cancelled") == 1, 1.0).otherwise(0.0))
    .alias("ewr_share_cancelled"),
)

table_ewr = (
    hour_spine(spark, WINDOW_START, WINDOW_END)
    .join(ewr_hourly, ["date", "hour"], how="left")
    .fillna(0, subset=["ewr_sched_arrivals", "ewr_sched_distance_mi",
                       "ewr_n_delay_observed"])
)

ewr_expected = WINDOW_DAYS * 24
assert table_ewr.count() == ewr_expected, "Newark grid is not complete"
print(f"{ewr_expected:,} Newark airport-hours")

13,128 Newark airport-hours


## Step 8 — Validation

Six checks. Each one catches a failure that would otherwise surface much later as a
plausible-looking but wrong number, at a point where it is expensive to trace.

In [17]:
# 1. The grid is complete and each key appears exactly once.
assert table_b.count() == EXPECTED_ROWS
assert table_b.dropDuplicates(["date", "hour", "airport"]).count() == EXPECTED_ROWS

# 2. No flight was lost or duplicated by the grouping and the joins.
assert table_b.agg(F.sum("sched_arrivals")).collect()[0][0] == flights_total
assert table_b.agg(F.sum("actual_arrivals")).collect()[0][0] == realised_total

# 3. Hours and airports are in range.
assert table_b.where(~F.col("hour").between(0, 23)).count() == 0
assert table_b.select("airport").distinct().count() == len(ARRIVAL_AIRPORTS)

# 4. Every share lies in [0, 1] where it is defined.
for column in ["share_longhaul", "share_cancelled", "share_delayed_15"]:
    out_of_range = table_b.where(
        F.col(column).isNotNull() & ~F.col(column).between(0, 1)
    ).count()
    assert out_of_range == 0, f"{column} out of range in {out_of_range} rows"

# 5. The delay columns are null in exactly the hours where nothing was observed.
mismatched = table_b.where(
    F.col("mean_arr_delay_min").isNull() != (F.col("n_delay_observed") == 0)
).count()
assert mismatched == 0, f"{mismatched} rows disagree on the delay denominator"

mismatched_ewr = table_ewr.where(
    F.col("ewr_mean_arr_delay_min").isNull() != (F.col("ewr_n_delay_observed") == 0)
).count()
assert mismatched_ewr == 0, f"{mismatched_ewr} Newark rows disagree on the denominator"

# 6. The neighbouring-hour columns are null only at the two ends of the window.
missing_neighbours = table_b.where(
    F.col("sched_arrivals_prev_hr").isNull()
    | F.col("sched_arrivals_next_hr").isNull()
).count()
assert missing_neighbours == 2 * len(ARRIVAL_AIRPORTS)

print("All checks passed.")

All checks passed.


In [18]:
# Descriptive summary, for the report and as a last look for anything absurd.
table_b.groupBy("airport").agg(
    F.sum("sched_arrivals").alias("scheduled"),
    F.sum("actual_arrivals").alias("landed"),
    F.avg("sched_arrivals").alias("mean_per_hour"),
    F.max("sched_arrivals").alias("max_arrivals_in_one_hour"),
    F.avg("share_cancelled").alias("mean_share_cancelled"),
    F.avg("mean_arr_delay_min").alias("mean_delay_min"),
    F.avg("share_longhaul").alias("mean_share_longhaul"),
).show(truncate=False)

# Zero-arrival hours are informative in their own right: they are the overnight
# curfew hours, and the model must reproduce them rather than smooth them away.
for airport in ARRIVAL_AIRPORTS:
    empty = table_b.where(
        (F.col("airport") == airport) & (F.col("sched_arrivals") == 0)
    ).count()
    print(f"{airport}: {empty} of {WINDOW_DAYS * 24} hours with nothing scheduled")

# The busiest hour *of the day*, which is what the report actually wants and
# what the previous column was mistaken for. Mean arrivals per clock hour,
# ranked, so the airport bank structure is visible rather than a single peak.
busiest = (
    table_b
    .groupBy("airport", "hour")
    .agg(F.avg("sched_arrivals").alias("mean_arrivals"))
    .withColumn(
        "rank",
        F.row_number().over(
            Window.partitionBy("airport").orderBy(F.desc("mean_arrivals"))
        ),
    )
    .where(F.col("rank") <= 3)
    .orderBy("airport", "rank")
)
busiest.show(truncate=False)

+-------+---------+------+------------------+------------------------+--------------------+-----------------+-------------------+
|airport|scheduled|landed|mean_per_hour     |max_arrivals_in_one_hour|mean_share_cancelled|mean_delay_min   |mean_share_longhaul|
+-------+---------+------+------------------+------------------------+--------------------+-----------------+-------------------+
|JFK    |193143   |188319|14.712294332723948|35                      |0.02085008571291058 |22.51175629186069|0.385407695864575  |
|LGA    |243611   |236351|18.55659658744668 |39                      |0.024921253520571007|17.54563075395143|0.03358131492961059|
+-------+---------+------+------------------+------------------------+--------------------+-----------------+-------------------+

JFK: 1910 of 13128 hours with nothing scheduled
LGA: 3432 of 13128 hours with nothing scheduled
+-------+----+------------------+----+
|airport|hour|mean_arrivals     |rank|
+-------+----+------------------+----+
|JFK  

## Step 9 — Write, and record which columns 2c may use

`flight_column_roles.json` is the leakage contract. `features` may be used as same-hour
model inputs; `realised_lag_only` may not, because none of it is knowable when the
decision is made. It is written as data rather than stated in a comment so that 2c can
assert against it — a naming convention is only as good as the memory of whoever writes
the next notebook, and this one has to survive being read by a marker as well.

In [19]:
COLUMN_ROLES = {
    "key": ["date", "hour", "airport"],
    "rows": EXPECTED_ROWS,
    "features": [
        "sched_arrivals",
        "sched_arrivals_prev_hr",
        "sched_arrivals_next_hr",
        "sched_distance_mi",
        "share_longhaul",
    ],
    "realised_lag_only": [
        "actual_arrivals",
        "arrival_slippage",
        "mean_arr_delay_min",
        "share_delayed_15",
        "share_cancelled",
        "n_delay_observed",
    ],
    "ewr_key": ["date", "hour"],
    "ewr_features": ["ewr_sched_arrivals", "ewr_sched_distance_mi"],
    "ewr_realised_lag_only": [
        "ewr_mean_arr_delay_min",
        "ewr_share_cancelled",
        "ewr_n_delay_observed",
    ],
    "note": (
        "Columns under realised_lag_only are outcomes of the hour they describe "
        "and must not be used as same-hour model inputs. They may enter the "
        "model only at a lag of at least one hour."
    ),
}

with open(CURATED_DIR / "flight_column_roles.json", "w") as handle:
    json.dump(COLUMN_ROLES, handle, indent=2)

# Every column written must be accounted for in exactly one role.
declared = set(COLUMN_ROLES["key"] + COLUMN_ROLES["features"]
               + COLUMN_ROLES["realised_lag_only"])
assert declared == set(table_b.columns), (
    f"Undeclared: {set(table_b.columns) - declared}, "
    f"declared but absent: {declared - set(table_b.columns)}"
)

# The Newark roles are asserted too. An unchecked declaration is worse than no
# declaration, because notebooks 2c and 4 trust this file.
ewr_declared = set(COLUMN_ROLES["ewr_key"] + COLUMN_ROLES["ewr_features"]
                   + COLUMN_ROLES["ewr_realised_lag_only"])
assert ewr_declared == set(table_ewr.columns), (
    f"Undeclared: {set(table_ewr.columns) - ewr_declared}, "
    f"declared but absent: {ewr_declared - set(table_ewr.columns)}"
)
print("Column roles recorded.")

Column roles recorded.


In [20]:
(
    table_b
    .orderBy("date", "hour", "airport")
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(CURATED_DIR / "flights_hourly.parquet"))
)

(
    table_ewr
    .orderBy("date", "hour")
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(CURATED_DIR / "flights_hourly_ewr.parquet"))
)

# Looked up by label rather than by position, so inserting a filter cannot
# silently make this report the wrong number.
dropped_corrupt = next(
    removed for label, _, removed in waterfall
    if label == "Plausible implied block time"
)

shapes = {
    "raw_bts_ingest": raw_total,
    "arrivals_jfk_lga": arrivals_total,
    "in_window_scheduled_arrivals": flights_total,
    "realised_arrivals": realised_total,
    "table_b_rows": EXPECTED_ROWS,
    "ewr_rows": ewr_expected,
    "arrivals_rolled_past_window_close": rolled_past_close,
    "arrivals_recovered_from_preceding_month": recovered,
    "rows_dropped_corrupt_block_time": dropped_corrupt,
}
with open(CURATED_DIR / "shapes_flights.json", "w") as handle:
    json.dump(shapes, handle, indent=2)

shapes

{'raw_bts_ingest': 1349639,
 'arrivals_jfk_lga': 461205,
 'in_window_scheduled_arrivals': 436754,
 'realised_arrivals': 424670,
 'table_b_rows': 26256,
 'ewr_rows': 13128,
 'arrivals_rolled_past_window_close': 49,
 'arrivals_recovered_from_preceding_month': 24,
 'rows_dropped_corrupt_block_time': 2}

In [21]:
spark.stop()

## What to carry into the report

- The waterfall in Step 3 populates the flight half of the preprocessing table. Quote the
  labels from `FILTERS` verbatim so the table and the code agree, as on the taxi side.
- The destination filter is worth one sentence of justification in the preprocessing
  section, because it is doing two jobs: selecting arrivals, and guaranteeing the clock
  fields are New York local.
- The share of arrivals shifted overnight, from Step 2, is the evidence that the date
  correction is real rather than decorative. One clause is enough.
- The boundary under-count on 1 January 2023 is a limitation to state plainly if the
  December 2022 file is not downloaded, with the measured figure attached.
- The zero-scheduled-arrival hours in Step 8 are worth a sentence: they are why the
  spine exists and why a count model is the right family for the target.

**Next:** notebook 2c — join Table A, Table B, Newark, and weather on the airport-hour
key, then build the temporal, holiday, and lag features. Two things to settle first:

1. **The target.** `n_pickups` is the clean count model and the Poisson justification is
   airtight, but the audience is a driver, and "queue at LGA at 22:00" is only good advice
   if those trips pay. Table A already carries `mean_fare` and `share_flat_fare` per hour,
   so demand and fare composition can be modelled as two layers — which gives the two
   models genuinely different jobs rather than two algorithms racing on one target.
2. **The 2a rate code filter**, which removes 5.35% of raw records by silently dropping
   nulls along with the undocumented code 99. That biases `n_pickups` non-uniformly, so
   it is cheaper to fix before the models are fitted than after.